# N-gram

In [ ]:
import collections

# 示例语料库corpus，与上方案例讲解中的语料库保持一致
corpus = "datawhale agent learns datawhale agent works"
tokens = corpus.split()
total_tokens = len(tokens)
print(f"总词数：{total_tokens}, 语料库：{tokens}")

# --- 第一步:计算 P(datawhale) ---
count_datawhale = tokens.count('datawhale')
p_datawhale = count_datawhale / total_tokens
print(f"第一步: P(datawhale) = {count_datawhale}/{total_tokens} = {p_datawhale:.3f}")

# --- 第二步:计算 P(agent|datawhale) ---
# 先计算 bigrams 用于后续步骤   bi是指2个词
bigrams = zip(tokens, tokens[1:])  # 生成词对 例如: (datawhale, agent)
bigram_counts = collections.Counter(bigrams)
count_datawhale_agent = bigram_counts[('datawhale', 'agent')]
# count_datawhale 已在第一步计算
p_agent_given_datawhale = count_datawhale_agent / count_datawhale
print(f"第二步: P(agent|datawhale) = {count_datawhale_agent}/{count_datawhale} = {p_agent_given_datawhale:.3f}")

# --- 第三步:计算 P(learns|agent) ---
count_agent_learns = bigram_counts[('agent', 'learns')]
count_agent = tokens.count('agent')
p_learns_given_agent = count_agent_learns / count_agent
print(f"第三步: P(learns|agent) = {count_agent_learns}/{count_agent} = {p_learns_given_agent:.3f}")

# --- 最后:将概率连乘 ---
p_sentence = p_datawhale * p_agent_given_datawhale * p_learns_given_agent
print(f"最后: P('datawhale agent learns') ≈ {p_datawhale:.3f} * {p_agent_given_datawhale:.3f} * {p_learns_given_agent:.3f} = {p_sentence:.3f}")


总词数：6, 语料库：['datawhale', 'agent', 'learns', 'datawhale', 'agent', 'works']
第一步: P(datawhale) = 2/6 = 0.333
第二步: P(agent|datawhale) = 2/2 = 1.000
第三步: P(learns|agent) = 1/2 = 0.500
最后: P('datawhale agent learns') ≈ 0.333 * 1.000 * 0.500 = 0.167


## N-gram 模型虽然简单有效，但有两个致命缺陷：

1.数据稀疏性 (Sparsity) ：如果一个词序列从未在语料库中出现，其概率估计就为 0，这显然是不合理的。虽然可以通过平滑 (Smoothing) 技术缓解，但无法根除。

2.泛化能力差：模型无法理解词与词之间的语义相似性。例如，即使模型在语料库中见过很多次 agent learns，它也无法将这个知识泛化到语义相似的词上。当我们计算 robot learns 的概率时，如果 robot 这个词从未出现过，或者 robot learns 这个组合从未出现过，模型计算出的概率也会是零。模型无法理解 agent 和 robot 在语义上的相似性。

# 二、神经网络语言模型与词嵌入(Word Embedding)

In [3]:
import numpy as np  # numpy是一个用于数值计算的 Python 库，这里用于处理向量和矩阵

# 假设我们已经学习到了简化的二维词向量
embeddings = {
    "king": np.array([0.9, 0.8]),  # []是python中的列表，它的函数的工具很少  => np.array([])是numpy中的数组，用于将列表转换成numpy数组，它的函数很多
    "queen": np.array([0.9, 0.2]),  # 类似java中的 int => Integer
    "man": np.array([0.7, 0.9]),
    "woman": np.array([0.7, 0.3])
}

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)   # np.dot()是numpy中的函数，用于计算两个数组的点积
    norm_product = np.linalg.norm(vec1) * np.linalg.norm(vec2)   # np.linalg.norm()是numpy中的函数，用于计算向量的模
    return dot_product / norm_product

# king - man + woman
result_vec = embeddings["king"] - embeddings["man"] + embeddings["woman"]

# 计算结果向量与 "queen" 的相似度
sim = cosine_similarity(result_vec, embeddings["queen"])

print(f"king - man + woman 的结果向量: {result_vec}")
print(f"该结果与 'queen' 的相似度: {sim:.4f}")


king - man + woman 的结果向量: [0.9 0.2]
该结果与 'queen' 的相似度: 1.0000


神经网络语言模型通过词嵌入，成功解决了 N-gram 模型的泛化能力差的问题。然而，它仍然有一个类似 N-gram 的限制：上下文窗口是固定的。它只能考虑固定数量的前文，这为能处理任意长序列的循环神经网络埋下了伏笔。

## 三、循环神经网络 (RNN) 与长短时记忆网络 (LSTM)

## 加载千问模型与分词器

In [1]:
import os

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer



# 指定模型ID
model_id = "Qwen/Qwen1.5-0.5B-Chat"

# 设置设备，优先使用GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 加载模型，并将其移动到指定设备
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    use_safetensors=True
).to(device)

print("模型和分词器加载完成！")

d:\agents\agent161\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


d:\agents\agent161\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sqcyn\.cache\huggingface\hub\models--Qwen--Qwen1.5-0.5B-Chat. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 1741.19it/s]


模型和分词器加载完成！


我们来创建一个对话提示，Qwen1.5-Chat 模型遵循特定的对话模板。然后，可以使用上一步加载的 tokenizer 将文本提示转换为模型能够理解的数字 ID（即 Token ID）。

In [7]:
# 准备对话输入
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "你好，在当今AI盛行的情况下，赚钱的路径有哪些？"}
]

# 使用分词器的模板格式化输入
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 编码输入文本
model_inputs = tokenizer([text], return_tensors="pt").to(device)

print("编码后的输入文本:")
print(model_inputs)


编码后的输入文本:
{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 108386,  96050, 106850,  15469,
         116203, 104248,   3837, 104518,   9370,  76837, 104719,  11319, 151645,
            198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]])}


现在可以调用模型的 generate() 方法来生成回答了。模型会输出一系列 Token ID，这代表了它的回答。

最后，我们需要使用分词器的 decode() 方法，将这些数字 ID 翻译回人类可以阅读的文本。

In [8]:
# 使用模型生成回答
# max_new_tokens 控制了模型最多能生成多少个新的Token
generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
)

# 将生成的 Token ID 截取掉输入部分
# 这样我们只解码模型新生成的部分
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# 解码生成的 Token ID
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n模型的回答:")
print(response)



模型的回答:
目前，你可以考虑以下几种赚钱方式：

1. 自然语言处理：通过自然语言处理技术，你可以开发聊天机器人、搜索引擎、自动问答系统等。

2. 机器学习：利用机器学习技术，可以实现自动化任务，比如识别图像、分析文本等。

3. 创造性思维：利用创造性思维进行创新工作，比如开发新型技术、设计新产品等。

4. 技术转让：将技术转让给其他公司或个人，以获得额外收入。

5. 风险投资：寻找和投资新兴科技领域，从中获取资本收益。

6. 智能化制造：在制造业中，利用智能化技术和设备来提高生产效率，降低成本，提高产品质量。


模型的回答:
KPI（Key Performance Indicators）是指衡量组织或项目成功的关键绩效指标，它可以用来评估组织的工作效率、客户满意度、投资回报率等，有助于了解组织的总体绩效。例如，你可以用“销售业绩”、“产品质量”、“市场份额”等作为KPI指标来衡量一个公司的成功程度。

## transformer实现


In [1]:
import torch
import torch.nn as nn   # 导入神经网络模块
import math

# --- 占位符模块，将在后续小节中实现 ---
class PositionalEncoding(nn.Module):
    """
    位置编码模块: 为每个词元添加位置信息，帮助模型理解序列的上下文。
    位置编码模块的输出与输入相同，只是在每个词元上添加了位置编码。
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 创建一个足够长的位置编码矩阵
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        # pe (positional encoding) 的大小为 (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # 偶数维度使用 sin, 奇数维度使用 cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # 将 pe 注册为 buffer，这样它就不会被视为模型参数，但会随模型移动（例如 to(device)）
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        """
            位置编码模块的前向传播函数, 为每个词元添加位置编码。
        """
        # x.size(1) 是当前输入的序列长度
        # 将位置编码加到输入向量上
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制模块
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"

        self.d_model = d_model  # 模型维度
        self.num_heads = num_heads  # 头数
        self.d_k = d_model // num_heads  # 每个头的维度

        # 定义 Q, K, V 和输出的线性变换层
        self.W_q = nn.Linear(d_model, d_model)  # 查询线性变换层
        self.W_k = nn.Linear(d_model, d_model)  # 键线性变换层
        self.W_v = nn.Linear(d_model, d_model)  # 值线性变换层
        self.W_o = nn.Linear(d_model, d_model)  # 输出线性变换层

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
          计算缩放点积注意力: 
        """
        # 1. 计算注意力得分 (QK^T)
        # 注意：这里 QK^T 是 (batch_size, num_heads, seq_length, seq_length) 形状的张量
        # 我们需要对每个头进行归一化，所以需要除以 sqrt(d_k) ,这是为了确保注意力得分在 -1 到 1 之间，从而保持注意力的稳定性
        # 同时，我们还需要将注意力得分的维度从 (batch_size, num_heads, seq_length, seq_length) 变换为 (batch_size, seq_length, seq_length)
        # 这是为了后续的 softmax 操作
        # torch.matmul()表示矩阵乘法，这里用于计算 QK^T，其中 Q 是查询矩阵，K 是键矩阵，^T 表示 K 的转置矩阵
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 2. 应用掩码 (如果提供) ： 掩码形状为 (batch_size, seq_length, seq_length) 的二进制张量
        # 我们需要将掩码应用到注意力得分上，以避免模型关注到不应该关注的位置
        # 掩码中为 0 的位置对应于模型不应该关注的位置，为 1 的位置对应于模型应该关注的位置
        # 掩码是通过 self-attention 计算的，这是一种常见的注意力机制，用于在处理序列数据时避免模型关注到不应该关注的位置
        if mask is not None:
            # 将掩码中为 0 的位置设置为一个非常小的负数，这样 softmax 后会接近 0
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # 3. 计算注意力权重 (Softmax)，将注意力得分归一化为概率分布
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # 4. 加权求和 (权重 * V)
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        # 将输入 x 的形状从 (batch_size, seq_length, d_model)即(批量大小, 序列长度, 模型维度) 例如: (1, 5, 128)
        # 变换为 (batch_size, num_heads, seq_length, d_k) 即(批量大小, 头数, 序列长度, 每个头的维度), 例如: (1, 8, 5, 16)
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) 
        # 这里的 transpose(1, 2) 是为了将头数和序列长度交换，方便后续的计算

    def combine_heads(self, x):
        # 将输入 x 的形状从 (batch_size, num_heads, seq_length, d_k)即(批量大小, 头数, 序列长度, 每个头的维度) 例如: (1, 8, 5, 16)
        # 变换为 (batch_size, seq_length, d_model) 即(批量大小, 序列长度, 模型维度) 例如: (1, 5, 128)
        batch_size, num_heads, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        # 这里的 transpose(1, 2) 是为了将头数和序列长度交换，方便后续的计算
        # 这里的 contiguous() 是为了将张量的内存布局设置为连续的，方便后续的操作
        # 这里的 view() 是为了将张量的形状从 (batch_size, num_heads, seq_length, d_k) 变换为 (batch_size, seq_length, d_model)

    def forward(self, Q, K, V, mask=None):
        # 1. 对 Q, K, V 进行线性变换
        Q = self.split_heads(self.W_q(Q))  # 对查询矩阵进行多头注意力机制，将查询矩阵的维度从 (batch_size, seq_length, d_model) 变换为 (batch_size, num_heads, seq_length, d_k) 即(批量大小, 头数, 序列长度, 每个头的维度) 例如: (1, 8, 5, 16)
        K = self.split_heads(self.W_k(K))  # 对键矩阵进行多头注意力机制
        V = self.split_heads(self.W_v(V))  # 对值矩阵进行多头注意力机制

        # 2. 计算缩放点积注意力
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)

        # 3. 合并多头输出并进行最终的线性变换
        output = self.W_o(self.combine_heads(attn_output))
        return output

    # 给个简单的计算案例如下
    # 假设我们有一个查询矩阵 Q，键矩阵 K，值矩阵 V，以及一个掩码 mask
    # 我们需要计算 QK^T，然后应用掩码，最后计算注意力权重 (Softmax)，并加权求和 (权重 * V)
    # 计算 QK^T
    # QK^T = (Q1, Q2, ..., Qn) * (K1^T, K2^T, ..., Kn^T)
    # 计算注意力得分
    # attn_scores = QK^T / sqrt(d_k)
    # 应用掩码
    # attn_scores = attn_scores * mask
    # 计算注意力权重
    # attn_probs = softmax(attn_scores, dim=-1)
    # 加权求和
    # output = attn_probs * V
    # 返回输出

class PositionWiseFeedForward(nn.Module):
    """
    位置前馈网络模块: 用于处理序列中的短距离依赖关系。
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionWiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x 形状: (batch_size, seq_len, d_model)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        # 最终输出形状: (batch_size, seq_len, d_model)
        return x

# --- 编码器核心层 ---
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads) # 待实现 多头自注意力
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout) # 待实现 位置前馈网络
        self.norm1 = nn.LayerNorm(d_model) # 层归一化（归一化指的是将每个维度的输出进行归一化）  1）将每个维度的输出进行归一化，使每个维度的输出在0到1之间，且每个维度的输出的均值为0，方差为1。
        self.norm2 = nn.LayerNorm(d_model) # 层归一化  , 用于归一化位置前馈网络的输出
        self.dropout = nn.Dropout(dropout) # 用于防止过拟合，随机将部分神经元设为0，减少依赖性

    def forward(self, x, mask):
        # 残差连接与层归一化将在 3.1.2.4 节中详细解释
        # 1. 多头自注意力
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        # 2. 前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

# --- 解码器核心层 ---
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads) # 待实现
        self.cross_attn = MultiHeadAttention(d_model, num_heads) # 待实现
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout) # 待实现
        self.norm1 = nn.LayerNorm(d_model) #对多头自注意力的输出进行归一化，使每个维度的输出在0到1之间，且每个维度的输出的均值为0，方差为1。
        self.norm2 = nn.LayerNorm(d_model) # 对交叉注意力的输出进行归一化，使每个维度的输出在0到1之间，且每个维度的输出的均值为0，方差为1。
        self.norm3 = nn.LayerNorm(d_model) # 对位置前馈网络的输出进行归一化，使每个维度的输出在0到1之间，且每个维度的输出的均值为0，方差为1。
        self.dropout = nn.Dropout(dropout) # 用于防止过拟合，随机将部分神经元设为0，减少依赖性

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        # 1. 掩码多头自注意力 (对自己)
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))

        # 2. 交叉注意力 (对编码器输出)
        cross_attn_output = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))

        # 3. 前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x


# --- 编码器-解码器模型 ---
class Encoder( nn.Module ):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_len):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        """
        编码器模型的前向传播函数, 用于编码输入序列。
        """
        x = self.embedding(x)   # 对输入序列进行词嵌入编码
        x = self.pos_encoder(x)   # 对词嵌入向量添加位置编码
        for layer in self.layers:  # 循环遍历每个编码器层，对输入序列进行编码
            x = layer(x, mask)   # 对输入序列进行编码
        return self.norm(x)     # 对编码后的输出进行层归一化

# --- 解码器模型 ---
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_len):
        """
        解码器模型的初始化函数, 用于初始化解码器模型的参数。
        :param vocab_size: 词汇表大小
        :param d_model: 模型维度
        :param num_layers: 层数
        :param num_heads: 多头注意力头数
        :param d_ff: 前馈网络维度
        :param dropout:  dropout 率: 用于防止过拟合，随机将部分神经元设为0，减少依赖性 
        :param max_len: 最大序列长度
               """
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.embedding(x)     # 对输入序列进行词嵌入编码
        x = self.pos_encoder(x)   # 对词嵌入向量添加位置编码
        for layer in self.layers:  # 循环遍历每个解码器层，对输入序列进行解码
            x = layer(x, encoder_output, src_mask, tgt_mask)   # 对输入序列进行解码
        return self.norm(x)     # 对解码后的输出进行层归一化


# -- 构建transformer模型 ----
# 它由 Encoder和Decoder组成
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_len=5000):
        super(Transformer, self).__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_len)  # 初始化编码器
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_len)  # 初始化解码器
        self.final_linear = nn.Linear(d_model, tgt_vocab_size)  # 初始化全连接层
        self.softmax = nn.LogSoftmax(dim=1)  # 初始化softmax层

    def generate_mask(self, src, tgt):
        """
        生成掩码函数, 用于生成编码器和解码器的掩码。
        :param src: 输入序列
        :param tgt: 目标序列
        :return: 编码器掩码, 解码器掩码
        """
        # src_mask: (batch_size, 1, 1, src_len)
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)   # (batch_size, 1, 1, src_len)
        
        # tgt_mask: (batch_size, 1, tgt_len, tgt_len)
        tgt_pad_mask = (tgt != 0).unsqueeze(1).unsqueeze(2) # (batch_size, 1, 1, tgt_len)
        tgt_len = tgt.size(1)
        # 下三角矩阵，用于防止看到未来的 token
        tgt_sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=src.device)).bool() # (tgt_len, tgt_len)
        tgt_mask = tgt_pad_mask & tgt_sub_mask

        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)  # 生成掩码
        
        encoder_output = self.encoder(src, src_mask)  # 对输入序列进行编码, 输出编码后的向量
        decoder_output = self.decoder(tgt, encoder_output, src_mask, tgt_mask) # 对目标序列进行解码, 输出解码后的向量
        
        output = self.final_linear(decoder_output)  # 对解码后的向量进行全连接层, 输出最终的预测
        output1 = self.softmax(output)  # 对最终的预测进行softmax层, 输出概率分布
        return output,output1

In [3]:
#测试模型
#1. 定义参数
src_vocab_size=5000  # 输入序列的词汇表大小
tgt_vocab_size=5000  # 目标序列的词汇表大小
d_model=512  # 模型的维度
num_layers=6  # 编码器和解码器的层数
num_heads=8  # 多头注意力的头数
d_ff=2048  # 前馈层的维度 , 用于模型的非线性变换
dropout=0.1  #  dropout 率, 用于防止过拟合,表示随机的将某些神经元设为0
max_len=100  # 最大序列长度, 用于生成掩码

#2. 定义模型
model = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_len)


# 3. 创建模拟输入数值
# 假设 batch_size=2, src_seq_len=10, tgt_seq_len=12
src=torch.randint(1, src_vocab_size, (2, 10))    # (batch_size, src_seq_len)表示 输入序列的词汇表索引
tgt=torch.randint(1, tgt_vocab_size, (2, 12))

# 模型前向传播
outputs=model( src,tgt )
output = outputs[0]

# 5. 打印输出
print("模型输出:", output.shape )
# 预期输出: torch.Size([2, 12, 5000]) -> (batch_size, tgt_seq_len, tgt_vocab_size)
print("**"*100)
print( output) 

模型输出: torch.Size([2, 12, 5000])
********************************************************************************************************************************************************************************************************
tensor([[[ 0.0208, -0.0499,  0.7329,  ...,  0.5460,  0.5559,  0.7882],
         [-0.1499,  0.7895,  1.2315,  ...,  0.2070,  0.4385, -0.0640],
         [-0.0352,  0.2783,  0.9175,  ...,  0.6342,  1.0711,  0.3885],
         ...,
         [-0.3660,  0.2772,  0.5482,  ...,  0.4308, -0.1113,  0.4759],
         [ 0.6833,  0.7406,  1.3620,  ...,  0.9060,  1.0896, -0.3149],
         [-0.0072,  1.0402,  0.7071,  ...,  0.8474,  0.9195,  0.3949]],

        [[ 0.4081,  0.9628,  1.3075,  ...,  0.8154, -0.1529,  0.2090],
         [ 0.2171,  0.8645,  0.8484,  ...,  1.2003,  0.0692,  0.2738],
         [ 1.0793,  0.7419,  0.8375,  ..., -0.0806,  0.1605,  0.3404],
         ...,
         [ 0.6548,  0.8048,  0.7925,  ...,  0.0976, -0.1495,  0.1400],
         [ 0.5157,  1.3156,